<h3 style='color:red'>Importing Data - Cleaning

In [3]:
import pandas as pd
import io

with open(r"C:\Users\ΚΩΣΤΑΣ\Desktop\DATA World\SQL\Superstore_Sales_Dataset\initial_dataset.csv", 'r', encoding='utf-8') as f:
    content = f.read()

# Αφαίρεσε τελικά semicolons και ξετύλιξε γραμμές με εισαγωγικά
lines = []
for line in content.split('\n'):
    line = line.rstrip(';').rstrip('\n')
    if line.startswith('"') and line.endswith('"'):
        line = line[1:-1].replace('""', '"')
    lines.append(line)

clean = '\n'.join(lines)
df = pd.read_csv(io.StringIO(clean))

# Διόρθωση τύπων
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce')
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True)



In [126]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9800 non-null   int64         
 1   Order ID       9800 non-null   object        
 2   Order Date     9800 non-null   datetime64[ns]
 3   Ship Date      9800 non-null   object        
 4   Ship Mode      9800 non-null   object        
 5   Customer ID    9800 non-null   object        
 6   Customer Name  9800 non-null   object        
 7   Segment        9800 non-null   object        
 8   Country        9800 non-null   object        
 9   City           9800 non-null   object        
 10  State          9800 non-null   object        
 11  Postal Code    9789 non-null   float64       
 12  Region         9800 non-null   object        
 13  Product ID     9800 non-null   object        
 14  Category       9800 non-null   object        
 15  Sub-Category   9800 n

Creating Dataframes - cust, prod - dimension dataframes, sales - fact dataframe

In [4]:
cust=df[['Customer ID', 'Customer Name', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Segment']]
cust=cust.drop_duplicates(subset='Customer ID')
prod=df[['Product ID', 'Product Name', 'Category', 'Sub-Category']]
prod=prod.drop_duplicates(subset='Product ID')
sales=df[['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Product ID', 'Sales']] 

<h3 style='color:red'>Basic Measures

In [135]:
print('Total Sales:', round(sales['Sales'].sum(), 2), '$')
print('Items sold:', sales['Order ID'].count())
print('Average Price:', round(sales['Sales'].mean(),2), '$')
total_orders=sales['Order ID'].nunique()
print('Total orders:', total_orders)
avg_order_amount = sales['Sales'].sum() / total_orders
print('Average Order Amount:', round(avg_order_amount,2), '$')
diferent_products_ordered=sales['Product ID'].nunique()
print('Different Products Ordered:', diferent_products_ordered)
print('Unique Customers:', sales['Customer ID'].nunique())

Total Sales: 2261536.78 $
Items sold: 9800
Average Price: 230.77 $
Total orders: 4922
Average Order Amount: 459.48 $
Different Products Ordered: 1861
Unique Customers: 793


<h3 style='color:red'>Customer Analysis


<h3 style='color:grey'>15 best customers -  more money spent

In [136]:

customer_sales=cust.merge(sales, how='left').groupby('Customer ID', as_index=False)['Sales'].sum()
best_customers = (customer_sales.merge(cust)
                  [['Customer ID', 'Customer Name', 'Sales', 'City']])
orders=cust.merge(sales, how='left').groupby('Customer ID', as_index=False)['Order ID'].nunique().rename(columns={'Order ID': 'Number_of_Orders'})
best_customers=best_customers.merge(orders, on='Customer ID').sort_values(by='Sales', ascending=False).head(15)
                  
best_customers

,Customer ID,Customer Name,Sales,City,Number_of_Orders
700,SM-20320,Sean Miller,25043.050,Monroe,5
741,TC-20980,Tamara Chand,19052.218,Seattle,5
621,RB-19360,Raymond Buch,15117.339,Auburn,6
730,TA-21385,Tom Ashbrook,14595.620,New York City,4
6,AB-10105,Adrian Barton,14473.571,Phoenix,10
434,KL-16645,Ken Lonsdale,14175.229,Chicago,12
669,SC-20095,Sanjit Chand,14142.334,Concord,9
327,HL-15040,Hunter Lopez,12873.298,Houston,6
683,SE-20110,Sanjit Engle,12209.438,New York City,11
131,CC-12370,Christopher Conant,12129.072,Fayetteville,5


<h3 style='color:grey'> Cities with most sales

In [178]:
city_sales=cust.merge(sales, how='left', on='Customer ID').groupby('City', as_index=False)['Sales'].sum()
num_of_sales = cust.merge(sales, how='left', on='Customer ID').groupby('City', as_index=False)['Order ID'].nunique()
city_sales.merge(num_of_sales, on='City').sort_values(by='Sales', ascending=False).head(10)

,City,Sales,Order ID
161,New York City,209428.6561,418
129,Los Angeles,139025.1150,348
183,Philadelphia,129953.1550,284
212,San Francisco,104842.2915,241
217,Seattle,104647.1798,183
95,Houston,87154.4058,179
33,Chicago,61106.1940,157
41,Columbus,39343.8790,134
151,Monroe,35707.0400,28
211,San Diego,35344.1270,79


<h3 style=grey> Same with above, different results because of data incocistency...

In [183]:
df.groupby('City').agg({'Sales':'sum', 'Order ID':'nunique'}).sort_values(by='Sales', ascending=False)

,Sales,Order ID
City,,
New York City,252462.547,439
Los Angeles,173420.181,378
Seattle,116106.322,210
San Francisco,109041.120,261
Philadelphia,108841.749,262
...,...,...
Ormond Beach,2.808,1
Pensacola,2.214,1
Jupiter,2.064,1


<h3 style=color:grey> Sales by customer segment

In [209]:
x1=cust.merge(sales, how='left', on='Customer ID').groupby('Segment').agg({'Sales':'sum', 'Order ID':'nunique', 'Segment':'count'})
x1 = x1.rename(columns={'Order ID':'Num_of_Orders', 'Segment':'Total_items'})
tot_sum=x1['Sales'].sum()
x1['percentage']=round(x1['Sales']/tot_sum * 100, 2)
x1

,Sales,Num_of_Orders,Total_items,percentage
Segment,,,,
Consumer,1.148061e+06,2537,5101,50.76
Corporate,6.884941e+05,1491,2953,30.44
Home Office,4.249822e+05,894,1746,18.79


<h3 style='color:grey'>Sales by Region

In [212]:
x1=cust.merge(sales, how='left', on='Customer ID').groupby('Region').agg({'Sales':'sum', 'Order ID':'nunique'})
x1 = x1.rename(columns={'Order ID':'Num_of_Orders'})
x1

,Sales,Num_of_Orders
Region,,
Central,514251.4722,1177
East,606351.1375,1372
South,396640.6213,811
West,744293.5517,1562


<h3 style='color:red'> Product Analysis

<h3 style='color:grey'> Products with most sales - top 15


In [15]:

x2=prod.merge(sales, how='left', on='Product ID')
x2=x2.groupby(['Product ID', 'Product Name', 'Category'], as_index=False).agg({'Sales':'sum', 'Order ID':'count'})
x2=x2.rename({'Order ID':'Total Orders'}).sort_values(by='Sales', ascending=False).head(15)
x2

,Product ID,Product Name,Category,Sales,Order ID
1613,TEC-CO-10004722,Canon imageCLASS 2200 Advanced Copier,Technology,61599.8240,5
776,OFF-BI-10003527,Fellowes PB500 Electric Punch Plastic Comb Bin...,Office Supplies,27453.3840,10
1641,TEC-MA-10002412,Cisco TelePresence System EX90 Videoconferenci...,Technology,22638.4800,1
80,FUR-CH-10002024,HON 5400 Series Task Chairs for Big and Tall,Furniture,21870.5760,8
691,OFF-BI-10001359,GBC DocuBind TL300 Electric Binding System,Office Supplies,19823.4790,11
657,OFF-BI-10000545,GBC Ibimaster 500 Manual ProClick Binding System,Office Supplies,19024.5000,9
1603,TEC-CO-10001449,Hewlett Packard LaserJet 3310 Copier,Technology,18839.6860,8
1630,TEC-MA-10001127,HP Designjet T520 Inkjet Large Format Printer ...,Technology,18374.8950,3
845,OFF-BI-10004995,GBC DocuBind P400 Electric Binding System,Office Supplies,17965.0680,6
1419,OFF-SU-10000151,High Speed Automatic Electric Letter Opener,Office Supplies,17030.3120,3


<h3 style='color:grey'> Sales by Category</h3>

In [293]:
pd.merge(prod, sales, how='left', on='Product ID').groupby('Category').agg({'Sales':'sum', 'Order ID':'count'})

,Sales,Order ID
Category,,
Furniture,728658.5757,2078
Office Supplies,705422.3340,5909
Technology,827455.8730,1813


<h3 style='color:grey'> Sales by Category and Subcategory</h3>

In [308]:
x3=pd.merge(prod, sales, how='left', on='Product ID').groupby(['Category', 'Sub-Category']).agg({'Sales':'sum', 'Order ID':'count'}).rename(columns={'Order ID':'Total Orders'})
x3['Sales Percentage'] = x3.groupby('Category')['Sales'].transform(lambda x: 100 * x / x.sum())
x3['Orders Percentage'] = x3.groupby('Category')['Total Orders'].transform(lambda x: 100 * x / x.sum())
x3.sort_values(['Category', 'Sales'], ascending=[True, False])

Sales  Total Orders  Sales Percentage  \
Category        Sub-Category                                                
Furniture       Chairs        322822.7310           607         44.303703   
                Tables        202810.6280           314         27.833424   
                Bookcases     113813.1987           226         15.619551   
                Furnishings    89212.0180           931         12.243322   
Office Supplies Storage       219343.3920           832         31.093911   
                Binders       200028.7850          1492         28.355891   
                Appliances    104618.4030           459         14.830605   
                Paper          76828.3040          1338         10.891107   
                Supplies       46420.3080           184          6.580499   
                Art            26705.4100           785          3.785734   
                Envelopes      16128.0460           248          2.286296   
                Labels         12347.7260           357          1.750402   
                Fasteners       3001.9600           214          0.425555   
Technology      Phones        327782.4480           876         39.613284   
                Machines      189238.6310           115         22.869936   
                Accessories   164186.7000           756         19.842351   
                Copiers       146248.0940            66         17.674428   

                              Orders Percentage  
Category        Sub-Category                     
Furniture       Chairs                29.210780  
                Tables                15.110683  
                Bookcases             10.875842  
                Furnishings           44.802695  
Office Supplies Storage               14.080217  
                Binders               25.249619  
                Appliances             7.767812  
                Paper                 22.643425  
                Supplies               3.113894  
                Art                   13.284820  
                Envelopes              4.196988  
                Labels                 6.041631  
                Fasteners              3.621594  
Technology      Phones                48.317705  
                Machines               6.343078  
                Accessories           41.698842  
                Copiers                3.640375

<h3 style='color:grey'>Bottom 25% products on sales</h3>

In [18]:
x4=pd.merge(prod, sales, how='left', on='Product ID').groupby(['Product ID', 'Product Name'], as_index=False).agg({'Sales':'sum', 'Order ID':'count'}).rename(columns={'Order ID':'Total Orders'})
x4[x4['Sales']<x4['Sales'].quantile(0.25)].sort_values(by='Sales')

,Product ID,Product Name,Sales,Total Orders
418,OFF-AP-10002203,Eureka Disposable Bags for Sanitaire Vibra Gro...,1.624,1
988,OFF-LA-10003388,Avery 5,5.760,1
1016,OFF-PA-10000048,Xerox 20,6.480,1
862,OFF-EN-10001535,Grip Seal Envelopes,7.072,1
1445,OFF-SU-10003936,Acme Serrated Blade Letter Opener,7.632,1
...,...,...,...,...
227,FUR-FU-10002506,"Tensor ""Hersey Kiss"" Styled Floor Lamp",93.528,3
165,FUR-FU-10000755,Eldon Expressions Mahogany Wood Desk Collection,93.600,2
193,FUR-FU-10001731,Acrylic Self-Standing Desk Frames,93.984,8
663,OFF-BI-10000756,Storex DuraTech Recycled Plastic Frosted Binders,94.128,8


<h3 style='color:red'>Time Analysis</h3>

<h3 style='color:grey'>Sales per Year</h3>

In [350]:
x5=sales.groupby(['Year']).agg({'Sales':'sum'})
x5['diff percent']=round((x5['Sales']-x5['Sales'].shift(1)) / x5['Sales'].shift(1) * 100,2)
x5

,Sales,diff percent
Year,,
2015,479856.2081,NaN
2016,459436.0054,-4.26
2017,600192.5500,30.64
2018,722052.0192,20.30


<h3 style='color:grey'>Sales per Year and Month</h3>

In [352]:
x6=sales.groupby(['Year', 'Month']).agg({'Sales':'sum'})
x6['diff percent']=round((x6['Sales']-x6['Sales'].shift(1)) / x6['Sales'].shift(1) * 100,2)
x6

Sales  diff percent
Year Month                           
2015 1       14205.7070           NaN
     2        4519.8920        -68.18
     3       55205.7970       1121.40
     4       27906.8550        -49.45
     5       23644.3030        -15.27
     6       34322.9356         45.16
     7       33781.5430         -1.58
     8       27117.5365        -19.73
     9       81623.5268        201.00
     10      31453.3930        -61.47
     11      77907.6607        147.69
     12      68167.0585        -12.50
2016 1       18066.9576        -73.50
     2       11951.4110        -33.85
     3       32339.3184        170.59
     4       34154.4685          5.61
     5       29959.5305        -12.28
     6       23599.3740        -21.23
     7       28608.2590         21.22
     8       36818.3422         28.70
     9       63133.6060         71.47
     10      31011.7375        -50.88
     11      75249.3995        142.65
     12      74543.6012         -0.94
2017 1       18542.4910        -75.13
     2       22978.8150         23.93
     3       51165.0590        122.66
     4       38679.7670        -24.40
     5       56656.9080         46.48
     6       39724.4860        -29.89
     7       38320.7830         -3.53
     8       30542.2003        -20.30
     9       69193.3909        126.55
     10      59583.0330        -13.89
     11      79066.4958         32.70
     12      95739.1210         21.09
2018 1       43476.4740        -54.59
     2       19920.9974        -54.18
     3       58863.4128        195.48
     4       35541.9101        -39.62
     5       43825.9822         23.31
     6       48190.7277          9.96
     7       44825.1040         -6.98
     8       62837.8480         40.18
     9       86152.8880         37.10
     10      77448.1312        -10.10
     11     117938.1550         52.28
     12      83030.3888        -29.60

<h3 style='color:grey'>Sales per Region per Year</h3>

In [368]:
x7=cust.merge(sales, how='left', on='Customer ID').groupby(['Region', 'Year']).agg({'Sales':'sum'})
x7['diff percent'] = round(x7.groupby('Region')['Sales'].diff()/x7.groupby('Region')['Sales'].shift(1) * 100,2)
x7

Sales  diff percent
Region  Year                           
Central 2015  105659.3801           NaN
        2016  103378.0361         -2.16
        2017  133787.3829         29.42
        2018  171426.6731         28.13
East    2015  102472.2860           NaN
        2016  132064.5428         28.88
        2017  169349.4378         28.23
        2018  202464.8709         19.55
South   2015  105543.9100           NaN
        2016   78135.2523        -25.97
        2017   96978.1050         24.12
        2018  115983.3540         19.60
West    2015  166180.6320           NaN
        2016  145858.1742        -12.23
        2017  200077.6243         37.17
        2018  232177.1212         16.04

<h3 style='color:grey'>Sales per Category  per Year</h3>

In [374]:
x8=prod.merge(sales, how='left', on='Product ID').groupby(['Category', 'Year']).agg({'Sales':'sum'})
x8['diff percent'] = round(x8.groupby('Category')['Sales'].diff()/x8.groupby('Category')['Sales'].shift(1) * 100,2)
x8

Sales  diff percent
Category        Year                           
Furniture       2015  156477.8811           NaN
                2016  164053.8674          4.84
                2017  195813.0400         19.36
                2018  212313.7872          8.43
Office Supplies 2015  149512.8200           NaN
                2016  133124.4070        -10.96
                2017  182417.5660         37.03
                2018  240367.5410         31.77
Technology      2015  173865.5070           NaN
                2016  162257.7310         -6.68
                2017  221961.9440         36.80
                2018  269370.6910         21.36

<h3 style='color:grey'>Sales per Category and Subcategory per Year</h3>

In [379]:
x9=prod.merge(sales, how='left', on='Product ID').groupby(['Category', 'Sub-Category','Year']).agg({'Sales':'sum'})
x9['diff percent'] = round(x9.groupby('Category')['Sales'].diff()/x9.groupby('Category')['Sales'].shift(1) * 100,2)
x9.head(50)

Sales  diff percent
Category        Sub-Category Year                          
Furniture       Bookcases    2015  20036.6776           NaN
                             2016  37476.7749         87.04
                             2017  26275.4665        -29.89
                             2018  30024.2797         14.27
                Chairs       2015  77046.4400        156.61
                             2016  70654.6730         -8.30
                             2017  81930.3450         15.96
                             2018  93191.2730         13.74
                Furnishings  2015  13636.9860        -85.37
                             2016  20525.2240         50.51
                             2017  26845.1160         30.79
                             2018  28204.6920          5.06
                Tables       2015  45757.7775         62.23
                             2016  35397.1955        -22.64
                             2017  60762.1125         71.66
                             2018  60893.5425          0.22
Office Supplies Appliances   2015  15160.7150           NaN
                             2016  23228.1790         53.21
                             2017  26016.7870         12.01
                             2018  40212.7220         54.56
                Art          2015   5897.5340        -85.33
                             2016   6091.6360          3.29
                             2017   5890.6080         -3.30
                             2018   8825.6320         49.83
                Binders      2015  43263.2670        390.20
                             2016  36049.7460        -16.67
                             2017  48994.5170         35.91
                             2018  71721.2550         46.39
                Envelopes    2015   3844.5900        -94.64
                             2016   4448.2180         15.70
                             2017   4456.6640          0.19
                             2018   3378.5740        -24.19
                Fasteners    2015    655.3880        -80.60
                             2016    545.2240        -16.81
                             2017    946.2740         73.56
                             2018    855.0740         -9.64
                Labels       2015   2825.4580        230.43
                             2016   2884.9120          2.10
                             2017   2792.6600         -3.20
                             2018   3844.6960         37.67
                Paper        2015  14332.7220        272.79
                             2016  14664.7620          2.32
                             2017  20326.0440         38.60
                             2018  27504.7760         35.32
                Storage      2015  49197.5260         78.87
                             2016  43321.2080        -11.94
                             2017  58751.9560         35.62
                             2018  68072.7020         15.86
                Supplies     2015  14335.6200        -78.94
                             2016   1890.5220        -86.81